# Inspect Vector DB Assets

Purpose:
    This notebook inspects the generated reference-retrieval assets in
    `llm_rag/iii_vector_db/` after the vector-db build step has run.

Flow:
1. Load the build summary, sparse corpus, parent contexts, and thresholds.
2. Display corpus and parent-context counts.
3. Inspect source and block-type distributions.
4. Preview the data that the retrieval engine uses at query time.


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    """Locate the repository root from a notebook working directory.

    Args:
        start (Path): Directory where the upward search should begin.

    Returns:
        Path: Repository root containing the `llm_rag/` folder.
    """
    for candidate in [start, *start.parents]:
        if (candidate / "llm_rag").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repo root containing 'llm_rag'.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
VECTOR_DB_DIR = REPO_ROOT / "llm_rag" / "iii_vector_db"

SUMMARY_PATH = VECTOR_DB_DIR / "build_summary.json"
CORPUS_PATH = VECTOR_DB_DIR / "reference_corpus.jsonl"
PARENT_CONTEXTS_PATH = VECTOR_DB_DIR / "parent_contexts.jsonl"
THRESHOLDS_PATH = VECTOR_DB_DIR / "thresholds.json"

print(f"Repo root: {REPO_ROOT}")
print(f"Vector DB dir: {VECTOR_DB_DIR}")


Repo root: G:\My Drive\Documents\3. Education\7. Imperial\Modules\SWE Group Project\IUCN_Reviewer
Vector DB dir: G:\My Drive\Documents\3. Education\7. Imperial\Modules\SWE Group Project\IUCN_Reviewer\llm_rag\iii_vector_db


In [2]:
def load_jsonl(path: Path) -> list[dict]:
    """Load a JSONL file into a list of dictionaries.

    Args:
        path (Path): JSONL file to load.

    Returns:
        list[dict]: Parsed non-empty JSONL rows.
    """
    rows = []
    with path.open("r", encoding="utf-8") as file_handle:
        for line in file_handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


build_summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
thresholds = json.loads(THRESHOLDS_PATH.read_text(encoding="utf-8"))
reference_corpus = load_jsonl(CORPUS_PATH)
parent_contexts = load_jsonl(PARENT_CONTEXTS_PATH)

build_summary


{'processed_dir': 'G:\\My Drive\\Documents\\3. Education\\7. Imperial\\Modules\\SWE Group Project\\RAG\\llm_rag\\ii_preprocessed_documents',
 'persist_dir': 'G:\\My Drive\\Documents\\3. Education\\7. Imperial\\Modules\\SWE Group Project\\RAG\\llm_rag\\iii_vector_db\\chroma_db\\reference_docs',
 'corpus_path': 'G:\\My Drive\\Documents\\3. Education\\7. Imperial\\Modules\\SWE Group Project\\RAG\\llm_rag\\iii_vector_db\\reference_corpus.jsonl',
 'parent_contexts_path': 'G:\\My Drive\\Documents\\3. Education\\7. Imperial\\Modules\\SWE Group Project\\RAG\\llm_rag\\iii_vector_db\\parent_contexts.jsonl',
 'collection_name': 'iucn_reference_docs',
 'embedding_model': 'BAAI/bge-m3',
 'raw_record_count': 1170,
 'chunk_count': 1743,
 'chunk_size': 800,
 'chunk_overlap': 100,
 'chroma_built': True,
 'table_row_count': 265,
 'fallback_table_row_count': 38,
 'synthetic_parent_count': 3}

In [3]:
pd.DataFrame(
    {
        "criterion": list(thresholds.get("criteria", {}).keys()),
        "title": [thresholds["criteria"][key].get("title") for key in thresholds.get("criteria", {}).keys()],
    }
)


,criterion,title
0,A,Population reduction
1,B,Geographic range
2,C,Small population size and decline
3,D,Very small or restricted population
4,E,Quantitative analysis


In [4]:
corpus_df = pd.DataFrame(reference_corpus)
corpus_df.head(10)


,text,search_text,doc_id,source_file,pdf_stem,page,block_type,table_id,table_title,row_id,parent_id,section_title,section_path,priority,fallback,table_name,chunk_id,display_text
0,Version 1.2 (July 2022) Citation: IUCN. 2022. ...,Source file: Guidelines_for_Reporting_Proporti...,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,1,text,None,None,NaN,None,Threatened,Threatened,1.0,False,None,Guidelines_for_Reporting_Proportion_Threatened...,Version 1.2 (July 2022) Citation: IUCN. 2022. ...
1,The uncertainty introduced by Data Deficient s...,Source file: Guidelines_for_Reporting_Proporti...,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,2,text,None,None,NaN,None,Guidelines for Reporting on Proportion Threate...,Guidelines for Reporting on Proportion Threate...,1.0,False,None,Guidelines_for_Reporting_Proportion_Threatened...,The uncertainty introduced by Data Deficient s...
2,". On the other hand, given that many DD specie...",Source file: Guidelines_for_Reporting_Proporti...,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,2,text,None,None,NaN,None,Guidelines for Reporting on Proportion Threate...,Guidelines for Reporting on Proportion Threate...,1.0,False,None,Guidelines_for_Reporting_Proportion_Threatened...,". On the other hand, given that many DD specie..."
3,Examining the fate of species formerly classif...,Source file: Guidelines_for_Reporting_Proporti...,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,2,text,None,None,NaN,None,Guidelines for Reporting on Proportion Threate...,Guidelines for Reporting on Proportion Threate...,1.0,False,None,Guidelines_for_Reporting_Proportion_Threatened...,Examining the fate of species formerly classif...
4,. Using all the available information on known...,Source file: Guidelines_for_Reporting_Proporti...,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,2,text,None,None,NaN,None,Guidelines for Reporting on Proportion Threate...,Guidelines for Reporting on Proportion Threate...,1.0,False,None,Guidelines_for_Reporting_Proportion_Threatened...,. Using all the available information on known...
5,"However, it is not immediately evident whether...",Source file: Guidelines_for_Reporting_Proporti...,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,2,text,None,None,NaN,None,Guidelines for Reporting on Proportion Threate...,Guidelines for Reporting on Proportion Threate...,1.0,False,None,Guidelines_for_Reporting_Proportion_Threatened...,"However, it is not immediately evident whether..."
6,. As a result of the uncertainty that Data Def...,Source file: Guidelines_for_Reporting_Proporti...,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,2,text,None,None,NaN,None,Guidelines for Reporting on Proportion Threate...,Guidelines for Reporting on Proportion Threate...,1.0,False,None,Guidelines_for_Reporting_Proportion_Threatened...,. As a result of the uncertainty that Data Def...
7, Lower bound: percentage of threatened specie...,Source file: Guidelines_for_Reporting_Proporti...,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,3,text,None,None,NaN,None,Guidelines for Reporting on Proportion Threate...,Guidelines for Reporting on Proportion Threate...,1.0,False,None,Guidelines_for_Reporting_Proportion_Threatened..

In [5]:
corpus_df.groupby(["source_file", "block_type"]).size().reset_index(name="count").sort_values(["source_file", "block_type"])


,source_file,block_type,count
0,Guidelines_for_Reporting_Proportion_Threatened...,text,22
1,Mapping_Standards_Version_1.20_Jan2024.pdf,table,14
2,Mapping_Standards_Version_1.20_Jan2024.pdf,table_row,72
3,Mapping_Standards_Version_1.20_Jan2024.pdf,text,147
4,RL_Standards_Consistency.pdf,table,6
5,RL_Standards_Consistency.pdf,table_row,44
6,RL_Standards_Consistency.pdf,text,325
7,RL_categories_and_criteria.pdf,table,12
8,RL_categories_and_criteria.pdf,table_row,17
9,RL_categories_and_criteria.pdf,text,138


In [6]:
parent_contexts_df = pd.DataFrame(parent_contexts)
parent_contexts_df.head(10)


,source_file,table_id,table_title,page,text,section_title,section_path,metadata
0,Mapping_Standards_Version_1.20_Jan2024.pdf,table_001,dec_lat The geographical latitude in decimal d...,11,Table title: dec_lat The geographical latitude...,dec_lat The geographical latitude in decimal d...,dec_lat The geographical latitude in decimal d...,{'csv_path': 'G:\My Drive\Documents\3. Educati...
1,Mapping_Standards_Version_1.20_Jan2024.pdf,table_002,4.1.2 Recommended polygon and point attributes,12,Table title: 4.1.2 Recommended polygon and poi...,4.1.2 Recommended polygon and point attributes,sens_comm > 4.1.2 Recommended polygon and poin...,{'csv_path': 'G:\My Drive\Documents\3. Educati...
2,Mapping_Standards_Version_1.20_Jan2024.pdf,table_003,Notes:,13,Table title: Notes:\nHeaders: Field | Descript...,Notes:,Notes:,{'csv_path': 'G:\My Drive\Documents\3. Educati...
3,Mapping_Standards_Version_1.20_Jan2024.pdf,table_004,Notes:,15,Table title: Notes:\nHeaders: column_1 | CODE ...,Notes:,Notes:,{'csv_path': 'G:\My Drive\Documents\3. Educati...
4,Mapping_Standards_Version_1.20_Jan2024.pdf,table_005,5.3 Seasonality Codes,16,Table title: 5.3 Seasonality Codes\nHeaders: c...,5.3 Seasonality Codes,Notes : > 5.3 Seasonality Codes,{'csv_path': 'G:\My Drive\Documents\3. Educati...
5,Mapping_Standards_Version_1.20_Jan2024.pdf,table_006,"5.4 Presence, Origin and Seasonality quick ref...",17,"Table title: 5.4 Presence, Origin and Seasonal...","5.4 Presence, Origin and Seasonality quick ref...","Notes: > 5.4 Presence, Origin and Seasonality ...",{'csv_path': 'G:\My Drive\Documents\3. Educati...
6,Mapping_Standards_Version_1.20_Jan2024.pdf,table_007,6.1 Countries with multiple distribution codes...,18,Table title: 6.1 Countries with multiple distr...,6.1 Countries with multiple distribution codes...,6 Countries of occurrence (COO) in SIS > 6.1 C...,{'csv_path': 'G:\My Drive\Documents\3. Educati...
7,Mapping_Standards_Version_1.20_Jan2024.pdf,table_008,7.1.1 How should EOO be calculated for polygon...,19,Table title: 7.1.1 How should EOO be calculate...,7.1.1 How should EOO be calculated for polygon...,6 Countries of occurrence (COO) in SIS > 7.1 E...,{'csv_path': 'G:\My Drive\Documents\3. Educati...
8,Mapping_Standards_Version_1.20_Jan2024.pdf,table_009,"11 Abbreviations, acronyms and definitions",30,"Table title: 11 Abbreviations, acronyms and de...","11 Abbreviations, acronyms and definitions","11 Abbreviations, acronyms and definitions",{'csv_path': 'G:\My Drive\Documents\3. Educati...
9,Mapping_Standards_Version_1.20_Jan2024.pdf,table_010,1.19 11/2020 RLTWG Multiple updates on documen...,31,Table title: 1.19 11/2020 RLTWG Multiple updat...,1.19 11/2020 RLTWG Multiple updates on documen...,Jemma Window / Ackbar Joolia > 1.19 11/2020 RL...,{'csv_path': 'G:\My Drive\Documents\3. Educati...
